In [103]:
import pandas as pd

df = pd.read_csv("final_dataset_with_line_numbers.csv")

In [105]:
df

,Unnamed: 0,file_path,contract_address,final_line_numbers,vuln,type,source_code,source
0,0,@c-layer/common/contracts/core/Core.sol,0x2a903c2f657803a2e614c42672247366d757ab34,{55},DC,train,pragma solidity ^0.6.0;\r\n\r\n\r\n\r\n\r\n\r\...,vulnerable_verified_dataset
1,1,A004.sol,0xcae22909c9dbc37c2f6c2780d1f58decd5d3b1fd,"{41, 46}","IOU,NC",train,pragma solidity ^0.4.25;\r\n\r\n\r\n\r\n/**\r\...,vulnerable_verified_dataset
2,2,ABIO_preICO.sol,0x6d84769b1e287a27f282a938c8110b22714dbf78,{201},TD,train,pragma solidity ^0.4.24;\r\ncontract Ownable{\...,vulnerable_verified_dataset
3,3,ACCURAL_DEPOSIT.sol,0x4320e6f8c05b27ab4707cd1f6d5ce6f3e4b3a5a1,"{49, 47}",RE,train,pragma solidity ^0.4.19;\r\n\r\ncontract ACCUR...,vulnerable_verified_dataset
4,4,ACL.sol,0x96f041b96708813b1d789606926c524e78543664,{252},DC,train,//File: contracts/acl/IACL.sol\r\npragma solid...,vulnerable_verified_dataset
...,...,...,...,...,...,...,...,...
795,795,unchecked_low_level_calls\etherpot_lotto.sol,NaN,"{109, 141}",unchecked_low_level_calls,NaN,/*\n * @source: https://github.com/etherpot/co...,smartbugs_curated
796,796,unchecked_low_level_calls\king_of_the_ether_th...,NaN,"{118, 132, 174, 110}",unchecked_low_level_calls,NaN,/*\n * @source: https://github.com/kieranelby/...,smartbugs_curated
797,797,unchecked_low_level_calls\lotto.sol,NaN,"{27, 20}",unchecked_low_level_calls,NaN,/*\n * @source: https://github.com/sigp/solidi...,smartbugs_curated
798,798,unchecked_low_level_calls\mishandled.sol,NaN,{14},unchecked_low_level_calls,NaN,/*\n * @source: https://github.com/seresistvan...,smartbugs_curated


In [107]:
max_length = 128

def identify_leeway():
    for leeway in range(1, 65):
        count_exceeding_max_length = 0
        for index, row in df.iterrows():
            line_numbers = list(eval(row['final_line_numbers']))
            line_numbers.sort()
            if len(line_numbers) > 1:
                min_num = line_numbers[0]
                max_num = line_numbers[-1]
                diff = max_num - min_num
                if diff > (max_length - leeway):
                    count_exceeding_max_length += 1
        print("leeway", str(leeway), "count_exceeding_max_length", count_exceeding_max_length)

identify_leeway()

leeway 1 count_exceeding_max_length 39
leeway 2 count_exceeding_max_length 40
leeway 3 count_exceeding_max_length 40
leeway 4 count_exceeding_max_length 40
leeway 5 count_exceeding_max_length 40
leeway 6 count_exceeding_max_length 40
leeway 7 count_exceeding_max_length 40
leeway 8 count_exceeding_max_length 41
leeway 9 count_exceeding_max_length 41
leeway 10 count_exceeding_max_length 41
leeway 11 count_exceeding_max_length 41
leeway 12 count_exceeding_max_length 41
leeway 13 count_exceeding_max_length 41
leeway 14 count_exceeding_max_length 42
leeway 15 count_exceeding_max_length 42
leeway 16 count_exceeding_max_length 43
leeway 17 count_exceeding_max_length 44
leeway 18 count_exceeding_max_length 44
leeway 19 count_exceeding_max_length 44
leeway 20 count_exceeding_max_length 46
leeway 21 count_exceeding_max_length 46
leeway 22 count_exceeding_max_length 46
leeway 23 count_exceeding_max_length 46
leeway 24 count_exceeding_max_length 46
leeway 25 count_exceeding_max_length 46
leeway 26

# Let's create the dataset with code snippets that can have upto 30 lines of leeway

In [110]:
def extract_snippet(solidity_code, target_lines, max_lines=64):
    """
    Extracts a code snippet from Solidity code that includes all target_lines.
    The snippet is expanded with extra context (if available) so that its total length
    does not exceed max_lines. The function returns:
      - A string containing the code snippet (each line as in the original file).
      - A dictionary mapping new snippet line numbers (as strings) to the original file line numbers.

    :param solidity_code: Solidity source code as a string.
    :param target_lines: List of line numbers that must be included in the snippet.
    :param max_lines: Maximum number of lines to include in the snippet.
    :return: A tuple (snippet_str, mapping_dict).
    """
    lines = solidity_code.splitlines()
    total_lines = len(lines)

    # Validate and adjust target line numbers (ensure they are within the file bounds)
    target_lines = [ln for ln in target_lines if 1 <= ln <= total_lines]
    if not target_lines:
        raise ValueError("None of the provided target lines are valid for the given file.")

    # Determine snippet boundaries (0-indexed)
    snippet_start = min(target_lines) - 1
    snippet_end = max(target_lines) - 1

    # Calculate initial snippet length
    snippet_length = snippet_end - snippet_start + 1

    # Expand snippet with context if it's shorter than max_lines
    if snippet_length < max_lines:
        extra_context = max_lines - snippet_length
        extra_before = extra_context // 2
        extra_after = extra_context - extra_before

        snippet_start = max(0, snippet_start - extra_before)
        snippet_end = min(total_lines - 1, snippet_end + extra_after)
    
    # Trim snippet if it exceeds max_lines after context extension
    if snippet_end - snippet_start + 1 > max_lines:
        snippet_end = snippet_start + max_lines - 1

    # Build the snippet and mapping dictionary
    snippet_lines = []
    mapping_dict = {}
    for idx, i in enumerate(range(snippet_start, snippet_end + 1)):
        snippet_lines.append(lines[i].rstrip())
        mapping_dict[str(i+1)] = idx + 1  # original line number (as string) -> new line number

    snippet_str = "\n".join(snippet_lines)
    new_line_numbers_list = []
    for line_number in target_lines:
        new_line_numbers_list.append(int(mapping_dict[str(line_number)]))
    return snippet_str, new_line_numbers_list

In [112]:
def map_target_lines(code, target_lines):
    # Split the code into lines.
    original_lines = code.splitlines()
    new_lines = []
    mapping = {}  # mapping from original line number to new line number

    # Process each original line (1-indexed).
    for i, line in enumerate(original_lines, start=1):
        if line.strip():  # if the line is not empty (or only whitespace)
            new_lines.append(line)
            mapping[i] = len(new_lines)  # record new line number
        else:
            mapping[i] = None  # line is removed (empty)

    # For each target line, get the corresponding new line number.
    new_target_lines = {}
    for orig in target_lines:
        new_num = mapping.get(orig)
        if new_num is not None:
            new_target_lines[orig] = new_num
        else:
            # Optionally, you might want to handle cases where the target line is empty.
            new_target_lines[orig] = "Removed (empty line)"
    return new_lines, new_target_lines

In [114]:
max_length = 128
leeway = 30

def create_dataset(df):
    new_rows = []
    more_lengthy = 0
    for index, row in df.iterrows():
        line_numbers = list(eval(row['final_line_numbers']))
        line_numbers.sort()
        
        source_code = row['source_code']
        original_source_code = source_code
        original_line_numbers = line_numbers
        
        new_code, new_target_mapping = map_target_lines(original_source_code, original_line_numbers)
        new_line_numbers_for_new_code = []
        for orig, new in new_target_mapping.items():
            if isinstance(new, int):
                new_line_numbers_for_new_code.append(new)
            else:
                print("Something wrong with line mapping after code compression by removing lines with whitespaces", new)
                continue
        line_numbers = new_line_numbers_for_new_code
        line_numbers.sort()

        source_code = "\n".join(new_code)
        source_code_lines = source_code.splitlines()
        if source_code_lines[-1].strip() == "":
            source_code_lines.pop()

        if len(source_code_lines) > 0:
            if len(line_numbers) > 1:
                min_num = line_numbers[0]
                max_num = line_numbers[-1]
                diff = max_num - min_num
                if diff < (max_length - leeway):
                    if len(source_code_lines) <= max_length:
                        new_row = [source_code, line_numbers, line_numbers, source_code, original_line_numbers, original_source_code]
                        new_rows.append(new_row)
                    else:
                        more_lengthy += 1
                        snippet, new_line_numbers_ = extract_snippet(source_code, line_numbers, max_lines=max_length)  
                        new_row = [snippet, new_line_numbers_, line_numbers, source_code, original_line_numbers, original_source_code]
                        new_rows.append(new_row)
            elif len(line_numbers) == 1:
                if len(source_code_lines) <= max_length:
                    new_row = [source_code, line_numbers, line_numbers, source_code, original_line_numbers, original_source_code]
                    new_rows.append(new_row)
                else:
                    more_lengthy += 1
                    snippet, new_line_numbers_ = extract_snippet(source_code, line_numbers, max_lines=max_length)
                    new_row = [snippet, new_line_numbers_, line_numbers, source_code, original_line_numbers, original_source_code]
                    new_rows.append(new_row)
        else:
            print("source code is empty")
            
    print(more_lengthy)
    return new_rows
                        
new_rows = create_dataset(df)   

213


In [116]:
len(new_rows)

755

In [118]:
from rich.console import Console
from rich.syntax import Syntax
from IPython.display import Markdown

def check_code_for_correctness(solidity_code, highlight_lines):
    console = Console()
    
    # Create a Syntax object with highlighting
    syntax = Syntax(solidity_code, "solidity", line_numbers=True, highlight_lines=highlight_lines)
    
    # Print formatted code with highlights
    console.print(syntax)

In [120]:
index = 0
def check_one_by_one():
    global index
    row = new_rows[index]
    print(row[2])
    check_code_for_correctness(row[0], row[1])
    check_code_for_correctness(row[3], row[2])
    check_code_for_correctness(row[5], row[4])
    index += 1

In [126]:
check_one_by_one()

[177]


   1              emit ICOStart(_value, weiPerABIO, minInvestment);                                                
   2          }                                                                                                    
   3          /**                                                                                                  
   4          * @notice Burns tokens leftover from an ICO round.                                                   
   5          * @dev This can be called by anyone after deadline since it's an essential and inevitable part.      
   6          */                                                                                                   
   7          function burnRestTokens() afterDeadline{                                                             
   8                  require(!restTokensBurned);                                                                  
   9                  abioToken.burnMyBalance();                                                                   
  10                  restTokensBurned = true;                                                                     
  11          }                                                                                                    
  12          function isRunning() view returns (bool){                                                            
  13              return (now < deadline);                                                                         
  14          }                                                                                                    
  15          function goalReached() internal;                                                                     
  16          modifier afterDeadline() { if (now >= deadline) _; }                                                 
  17 }                                                                                                             
  18 contract ABIO_preICO is ABIO_BaseICO{                                                                         
  19     address ICOAddress;                                                                                       
  20     ABIO_ICO ICO;                                                                                             
  21     uint finalDeadline;                                                                                       
  22     constructor(address _abioAddress, uint _lenInMins, uint _minWeiInvestment, address _treasury, uint _priceI
  23         treasury = _treasury;                                                                                 
  24         abioToken = ABIO_Token(_abioAddress);                                                                 
  25         weiPerABIO = _priceInWei;                                                                             
  26         fundingGoal = _goalInWei;                                                                             
  27         minInvestment = _minWeiInvestment;                                                                    
  28         startDate = now;                                                                                      
  29         length = _lenInMins * 1 minutes;                                                                      
  30      }                                                                                                        
  31      /**                                                                                                      
  32      * @notice Called by dev to supply the address of the ICO (which is created after the PreICO)             
  33      * @dev We check if `fundingGoal` is reached again, because this MIGHT be called after it is reached, so `
  34      */                                                                                                       
  35     function supplyICOContract(address _addr[38;2;

    1 pragma solidity ^0.4.24;                                                                                     
    2 contract Ownable{                                                                                            
    3     address public owner;                                                                                    
    4     event ownerTransfer(address indexed oldOwner, address indexed newOwner);                                 
    5     event ownerGone(address indexed oldOwner);                                                               
    6     constructor(){                                                                                           
    7         owner = msg.sender;                                                                                  
    8     }                                                                                                        
    9     modifier onlyOwner(){                                                                                    
   10         require(msg.sender == owner);                                                                        
   11         _;                                                                                                   
   12     }                                                                                                        
   13     function changeOwner(address _newOwner) public onlyOwner{                                                
   14         require(_newOwner != address(0x0));                                                                  
   15         emit ownerTransfer(owner, _newOwner);                                                                
   16         owner = _newOwner;                                                                                   
   17     }                                                                                                        
   18 }                                                                                                            
   19 contract Haltable is Ownable{                                                                                
   20     bool public paused;                                                                                      
   21     event ContractPaused(address by);                                                                        
   22     event ContractUnpaused(address by);                                                                      
   23     /**                                                                                                      
   24      * @dev Paused by default.                                                                               
   25      */                                                                                                      
   26     constructor(){                                                                                           
   27         paused = true;                                                                                       
   28     }                                                                                                        
   29     function pause() public onlyOwner {                                                                      
   30         paused = true;                                                                                       
   31         emit ContractPaused(owner);                                                                          
   32     }                                                                                                        
   33     function unpause() public onlyOwner {                                                                    
   34         paused = false;                                                                                      
   35         emit ContractUnpaused

    1 pragma solidity ^0.4.24;                                                                                     
    2 contract Ownable{                                                                                            
    3     address public owner;                                                                                    
    4     event ownerTransfer(address indexed oldOwner, address indexed newOwner);                                 
    5     event ownerGone(address indexed oldOwner);                                                               
    6                                                                                                              
    7     constructor(){                                                                                           
    8         owner = msg.sender;                                                                                  
    9     }                                                                                                        
   10     modifier onlyOwner(){                                                                                    
   11         require(msg.sender == owner);                                                                        
   12         _;                                                                                                   
   13     }                                                                                                        
   14     function changeOwner(address _newOwner) public onlyOwner{                                                
   15         require(_newOwner != address(0x0));                                                                  
   16         emit ownerTransfer(owner, _newOwner);                                                                
   17         owner = _newOwner;                                                                                   
   18     }                                                                                                        
   19 }                                                                                                            
   20 contract Haltable is Ownable{                                                                                
   21     bool public paused;                                                                                      
   22     event ContractPaused(address by);                                                                        
   23     event ContractUnpaused(address by);                                                                      
   24                                                                                                              
   25     /**                                                                                                      
   26      * @dev Paused by default.                                                                               
   27      */                                                                                                      
   28     constructor(){                                                                                           
   29         paused = true;                                                                                       
   30     }                                                                                                        
   31     function pause() public onlyOwner {                                                                      
   32         paused = true;                                                                                       
   33         emit ContractPaused(owner);                                                                          
   34     }                                                                                                        
   35     function unpause() public onlyOwner {         

In [128]:
final_rows = []
for row in new_rows:
    final_rows.append([row[0], row[1]])
# print(final_rows[0])
bug_localization_dataset_v1 = pd.DataFrame(final_rows)
bug_localization_dataset_v1.to_csv("bug_localization_dataset_v1.csv", index=False)